In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col, lit, sum as spark_sum, avg as spark_avg, max as spark_max, countDistinct, when, log2, expr

from common.const import (
    START_TIMESTAMP_5CORE_TRAIN, END_TIMESTAMP_5CORE_TRAIN,
    SCHEMA, TABLE_5CORE_TIMESTAMP_W_HIS_DICT, TABLE_5CORE_TIMESTAMP_W_HIS_TRAIN_USER_ITEM_INTERACTIONS,
    PARQUET_FOLDERS_5CORE_TIMESTAMP_W_HIS, TABLE_RAW_PRODUCTS, PARQUET_FOLDER_RAW_PRODUCTS, TABLE_RECENT_ITEM_REVIEW_SUMMARY,
    TABLE_FEATURED_DENSE_VECTORS, TABLE_USER_ITEM_STATE_SNAPSHOT,
    CLICKHOUSE_HOST, CLICKHOUSE_PASSWORD, CLICKHOUSE_PORT, CLICKHOUSE_USERNAME
)

from common.helper import get_categories, clickhouse_extract_data_as_df, clickhouse_extract_data_as_list, connect_clickhouse
import shutil
from tqdm import tqdm
import time
import gc # garbage collector to free up memory

categories = get_categories()
categories = categories[:10] # This line is for testing only (selecting a subset of categories).
LOADING_BATCH_SIZE = 3_000_000

per_user_embedding_table = "featured_user_vectors"
per_item_embedding_table = TABLE_FEATURED_DENSE_VECTORS

content_vector_name = 'content_vector'
cf_vector_name = 'cf_vector'

E0000 00:00:1773178244.976895 1477304 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773178244.976929 1477304 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1773178244.976931 1477304 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773178244.976932 1477304 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1773178244.976932 1477304 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [2]:
try:
    spark.stop()
except:
    pass

# Create Spark session
spark = SparkSession.builder \
    .appName("GroupB-RecSys") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .config("spark.kryoserializer.buffer", "64m") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "4g") \
    .getOrCreate()

# Disable Arrow-based columnar data transfers
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

# Set log level to reduce noise
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")
print("Spark session created successfully!")

26/03/10 21:31:32 WARN Utils: Your hostname, infb081tch011 resolves to a loopback address: 127.0.0.1; using 10.60.44.21 instead (on interface enp107s0)
26/03/10 21:31:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 21:31:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.0
Spark session created successfully!


In [3]:
def exclude_history_sql_where_clause(rec_user_id):
    return f"""NOT has(
        (
            SELECT interacted_items
            FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_TRAIN_USER_ITEM_INTERACTIONS}
            WHERE user_id = '{rec_user_id}'
        ),
        v.parent_asin
    )"""

# 1. Content-based Filtering

With content-based filtering, we don't filter the products users review in the last 6 months since users might still like products they buy a long time in the past.

In [ ]:
# --- Configuration ---
# 100 is very safe. You can likely lower this back to 10 once this works.
NUM_BATCHES = 100 
# ---------------------

client = connect_clickhouse()

client.command(f"DROP TABLE IF EXISTS {SCHEMA}.temp_user_profiles_join;")
client.command(f"ALTER TABLE {SCHEMA}.{per_user_embedding_table} DROP COLUMN IF EXISTS {content_vector_name};")
client.command(f"ALTER TABLE {SCHEMA}.{per_user_embedding_table} ADD COLUMN IF NOT EXISTS {content_vector_name} Array(Float32);")

print("Step 1: Creating temporary JOIN table...")
client.command(f"""
    CREATE TABLE IF NOT EXISTS {SCHEMA}.temp_user_profiles_join (
        user_id String, 
        {content_vector_name} Array(Float32)
    ) ENGINE = Join(ANY, LEFT, user_id);
""")

print("Step 2: Processing and inserting vectors in batches to avoid OOM...")
for batch_num in range(NUM_BATCHES):
    print(f"  -> Processing batch {batch_num + 1}/{NUM_BATCHES}...")
    
    insert_query = f"""
        INSERT INTO {SCHEMA}.temp_user_profiles_join
        WITH 
            -- 1. Isolate the exact batch of users FIRST
            FilteredTrain AS (
                SELECT user_id, parent_asin, rating
                FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT['train']}
                WHERE rating >= 4
                  AND cityHash64(user_id) % {NUM_BATCHES} = {batch_num}
            ),
            
            -- 2. ONLY load vectors for items that appear in this specific batch!
            -- This is the crucial step that stops the 10.8GB OOM crash.
            FilteredVec AS (
                SELECT parent_asin, mixed_vector
                FROM {SCHEMA}.{per_item_embedding_table}
                WHERE parent_asin IN (SELECT parent_asin FROM FilteredTrain)
            ),
            
            -- 3. Now perform the JOIN and math on the tiny subsets
            UserAggregates AS (
                SELECT
                    train.user_id,
                    sumMap(
                        range(toUInt32(length(vec.mixed_vector))), 
                        arrayMap(x -> x * toFloat32(train.rating), vec.mixed_vector)
                    ) AS sum_map_tuple,
                    sum(toFloat32(train.rating)) AS total_weight
                FROM FilteredTrain AS train
                JOIN FilteredVec AS vec
                    ON train.parent_asin = vec.parent_asin
                GROUP BY train.user_id
            )
            
        SELECT
            user_id,
            arrayMap(x -> x / total_weight, sum_map_tuple.2) AS {content_vector_name}
        FROM UserAggregates
        SETTINGS 
            join_algorithm = 'grace_hash',
            max_bytes_before_external_group_by = 8000000000;
    """
    client.command(insert_query)

print("Step 3: Triggering background ALTER TABLE UPDATE...")
alter_query = f"""
    ALTER TABLE {SCHEMA}.{per_user_embedding_table}
    UPDATE {content_vector_name} = joinGet('{SCHEMA}.temp_user_profiles_join', '{content_vector_name}', user_id)
    WHERE joinGet('{SCHEMA}.temp_user_profiles_join', '{content_vector_name}', user_id) IS NOT NULL;
"""
client.command(alter_query)

print("Step 4: Waiting for mutation to complete in the background...")
while True:
    pending_mutations = client.command(f"""
        SELECT count() 
        FROM system.mutations 
        WHERE database = '{SCHEMA}' 
          AND table = '{per_user_embedding_table}' 
          AND is_done = 0
    """)
    
    if pending_mutations == 0:
        print("Mutation completed successfully!")
        break
        
    print(f"  -> Mutation still running. Waiting 100 seconds...")
    time.sleep(100)

print("Step 5: Cleaning up temporary table...")
client.command(f"DROP TABLE IF EXISTS {SCHEMA}.temp_user_profiles_join;")

client.close()
print("All done!")

In [4]:
rec_user_id = "AE22236AFRRSMQIKGG7TPTB75QEA"

get_rec_user_embedding_query = f"""
SELECT {content_vector_name}
FROM {SCHEMA}.{per_user_embedding_table}
WHERE user_id = '{rec_user_id}'
"""
rec_user_embedding = clickhouse_extract_data_as_list(get_rec_user_embedding_query)[0][0]


content_based_query = f"""
WITH {rec_user_embedding} AS reference_vector

SELECT
    raw_products.parent_asin,
    raw_products.title,
    raw_products.product_category,
    rec_products.cosine_similarity
FROM {SCHEMA}.{TABLE_RAW_PRODUCTS} AS raw_products
JOIN (
        SELECT
            parent_asin,
            1 - cosineDistance(mixed_vector, reference_vector) AS cosine_similarity
        FROM {SCHEMA}.{per_item_embedding_table} v
        WHERE {exclude_history_sql_where_clause(rec_user_id)}
        ORDER BY cosine_similarity DESC
        LIMIT 10
    ) AS rec_products ON raw_products.parent_asin = rec_products.parent_asin
ORDER BY rec_products.cosine_similarity DESC
"""

content_based_recs = clickhouse_extract_data_as_df(content_based_query)
content_based_recs

,parent_asin,title,product_category,cosine_similarity
0,B019PZPYK6,Anker Powerline+ Micro USB (1ft) The Premium D...,Electronics,0.980743
1,B08CZ4HJC1,Dysania HTV Heat Transfer Vinyl Bundle - 40 Pa...,Arts_Crafts_and_Sewing,0.980126
2,B07XKS122R,NANW Screen Protector,Cell_Phones_and_Accessories,0.979656
3,B01GCIEAKO,"Cutebe Case for Galaxy S7, Shockproof Hard PC+...",Cell_Phones_and_Accessories,0.979567
4,B07DXPTCXP,"Fidget Toys Set, 36 Pack Sensory Tools Bundle ...",Toys_and_Games,0.979442
5,B0092PZZLS,iPad Mini 1/2 Leather Case in Black,Electronics,0.979241
6,B07PY26Y6Y,100 PCS Multipurpose Sewing Clips Perfect for ...,Arts_Crafts_and_Sewing,0.979135
7,B093WQBKZY,Dadanism All-New Kindle Fire HD 10 Tablet Case...,Electronics,0.979119
8,B07Q7GJSN9,"Punswan for Airpods 1 & 2 Charging Case,3D Cut...",Electronics,0.978856
9,B093GNV9M4,"Fidget Toys, Push Bubble Fidgets Sensory Toy, ...",Toys_and_Games,0.978698


# 2. Collaborative Filtering with ALS

For CF, we filter reviews from 5core users in the last 6 months (based on training data) to:
- Recommend products that are highly rated recently (since CF doesn't care about content)
- Reduce data size

We do that on a frequent basis (e.g., weekly) to update preferences of users, make it up-to-date.

## 2.1. Load data

In [33]:
def dump_split_to_parquet(split, output_dir, batch_size=LOADING_BATCH_SIZE):
    try:
        shutil.rmtree(output_dir)
        print(f"Successfully deleted {output_dir}")
    except OSError as e:
        print(f"Error: {e.filename} - {e.strerror}")
        
    os.makedirs(output_dir, exist_ok=True)

    item_col = "user_id"
    last_item = None
    batch_id = 0

    while True:
        query = f"""
        SELECT user_id, parent_asin, rating
        FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT[split]}
        WHERE 1=1
        {"AND timestamp <= " + str(END_TIMESTAMP_5CORE_TRAIN) + " AND timestamp >= " + str(START_TIMESTAMP_5CORE_TRAIN) if split == 'train' else ""}
        {"AND " + item_col + " > '" + last_item + "'" if last_item else ""}
        ORDER BY {item_col}
        LIMIT {batch_size}
        """

        pdf = clickhouse_extract_data_as_df(query)

        if pdf.empty:
            break

        file_path = os.path.join(output_dir, f"batch_{batch_id}.parquet")
        pdf.to_parquet(file_path, index=False)

        last_item = pdf[item_col].iloc[-1]
        batch_id += 1

    print(f"Finished dumping {split} data.")

for split, output_dir in PARQUET_FOLDERS_5CORE_TIMESTAMP_W_HIS.items():
    dump_split_to_parquet(split, output_dir)

train_5core = spark.read.parquet(PARQUET_FOLDERS_5CORE_TIMESTAMP_W_HIS["train"])
train_5core.cache()
print("Train 5core:", train_5core.count())
valid_5core = spark.read.parquet(PARQUET_FOLDERS_5CORE_TIMESTAMP_W_HIS["valid"])
valid_5core.cache()
print("Valid 5core:", valid_5core.count())
test_5core = spark.read.parquet(PARQUET_FOLDERS_5CORE_TIMESTAMP_W_HIS["test"])
print("Test 5core:", test_5core.count())

Error: /home/vuongtc/Documents/5core_timestamp_w_his_train_parquet - No such file or directory
Finished dumping train data.
Error: /home/vuongtc/Documents/5core_timestamp_w_his_valid_parquet - No such file or directory
Finished dumping valid data.
Error: /home/vuongtc/Documents/5core_timestamp_w_his_test_parquet - No such file or directory
Finished dumping test data.


Train 5core: 13993793


Valid 5core: 2060872
Test 5core: 2383816


In [34]:
def dump_raw_products_to_parquet(output_dir, batch_size=LOADING_BATCH_SIZE):
    try:
        shutil.rmtree(output_dir)
        print(f"Successfully deleted {output_dir}")
    except OSError as e:
        print(f"Error: {e.filename} - {e.strerror}")
        
    os.makedirs(output_dir, exist_ok=True)

    item_col = "parent_asin"
    last_item = None
    batch_id = 0

    while True:
        query = f"""
        SELECT parent_asin, title, average_rating, rating_number, categories, main_category, product_category
        FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
        {"WHERE " + item_col + " > '" + last_item + "'" if last_item else ""}
        ORDER BY {item_col}
        LIMIT {batch_size}
        """

        pdf = clickhouse_extract_data_as_df(query)

        if pdf.empty:
            break

        file_path = os.path.join(output_dir, f"batch_{batch_id}.parquet")
        pdf.to_parquet(file_path, index=False)

        last_item = pdf[item_col].iloc[-1]
        batch_id += 1

dump_raw_products_to_parquet(output_dir=PARQUET_FOLDER_RAW_PRODUCTS)

raw_products = spark.read.parquet(PARQUET_FOLDER_RAW_PRODUCTS)
# raw_products.show(5)

Error: /home/vuongtc/Documents/raw_products_parquet - No such file or directory


## 2.2. Convert IDs to Numeric Index

In [6]:
user_indexer = StringIndexer(
    inputCol="user_id",
    outputCol="user_idx",
    handleInvalid="skip"
)

item_indexer = StringIndexer(
    inputCol="parent_asin",
    outputCol="item_idx",
    handleInvalid="skip"
)

pipeline = Pipeline(stages=[user_indexer, item_indexer])

index_model = pipeline.fit(train_5core)

train_5core = index_model.transform(train_5core)
valid_5core = index_model.transform(valid_5core)
test_5core  = index_model.transform(test_5core)

## 2.3. Train ALS

In [7]:
ALS_RANK = 50
ALS_REGPARAM = 0.5

als = ALS(
    userCol="user_idx",
    itemCol="item_idx",
    ratingCol="rating",
    rank=ALS_RANK,
    maxIter=10,
    regParam=ALS_REGPARAM,
    coldStartStrategy="drop", # drop NaN predictions for cold-start items
    nonnegative=False # allow negative latent factors
)

model = als.fit(train_5core)

## 2.4. Validate Model (RMSE)

In [8]:
pred_valid = model.transform(valid_5core)

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse_valid = evaluator.evaluate(pred_valid)
print("Validation RMSE:", rmse_valid)

Validation RMSE: 1.426623754832779


## 2.5. Test Performance

In [9]:
pred_test = model.transform(test_5core)
rmse_test = evaluator.evaluate(pred_test)
print("Test RMSE:", rmse_test)

Test RMSE: 1.464801213937835


## 2.6. Hyperparameter Tuning

In [7]:
# 1. CRITICAL: Still absolutely required for ALS to prevent lineage OOM
SPARK_CHECKPOINT_DIR = os.path.join("/home", "vuongtc", "Documents", "spark_checkpoints")
os.makedirs(SPARK_CHECKPOINT_DIR, exist_ok=True)
spark.sparkContext.setCheckpointDir(SPARK_CHECKPOINT_DIR)

# Initialize the base ALS estimator
# We set the static parameters here, including our memory safeguard
als = ALS(
    userCol="user_idx",
    itemCol="item_idx",
    ratingCol="rating",
    maxIter=10,
    coldStartStrategy="drop",
    nonnegative=False,
    checkpointInterval=5  # Keeps the JVM heap from overflowing during each model's training
)

# Initialize the evaluator
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# 2. Build the parameter grid
param_grid = ParamGridBuilder() \
    .addGrid(als.rank, [10, 25, 50, 100]) \
    .addGrid(als.regParam, [0.01, 0.1, 0.5]) \
    .build()

# 3. Setup the CrossValidator
cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=5, # 5-fold cross-validation
    # MEMORY SAFEGUARD: Keep parallelism at 1 (the default). 
    # Do NOT increase this, or Spark will try to train multiple models at the exact same time, causing an instant OOM.
    parallelism=1 
)

print("Starting Cross-Validation... this will take a while!")

# 4. Fit the models (Spark handles the loop internally)
# Note: You pass your FULL training dataset here, CrossValidator handles the splitting
cv_model = cv.fit(train_5core)

# 1. Get the list of all evaluation metrics (e.g., RMSE scores)
metrics = cv_model.avgMetrics

# 2. Find the index of the best score (ALS uses RMSE, so we want the minimum)
best_idx = metrics.index(min(metrics))

# 3. Get the list of all parameter combinations tested
param_maps = cv_model.getEstimatorParamMaps()

# 4. Extract the winning parameter dictionary using our index
best_params = param_maps[best_idx]

# 5. Print out the winning combination cleanly
print("*** Best Model Parameters ***")
for param, value in best_params.items():
    print(f"{param.name}: {value}")

# You can also print the best metric while you are at it!
print(f"\nBest Validation RMSE: {min(metrics)}\n")


# Print out performance of all runs
# Get the list of all evaluation metrics and parameter maps
metrics = cv_model.avgMetrics
param_maps = cv_model.getEstimatorParamMaps()

print("=== All Hyperparameter Combinations and Validation RMSE ===\n")

# Zip them together and loop through
for metric, params in zip(metrics, param_maps):
    # Format the parameters into a readable string
    param_str = ", ".join([f"{p.name}: {v}" for p, v in params.items()])
    
    # Print the metric and the corresponding parameters
    print(f"RMSE: {metric:.4f}  |  {param_str}")

Starting Cross-Validation... this will take a while!


*** Best Model Parameters ***
rank: 50
regParam: 0.5

Best Validation RMSE: 1.3041832846005488

=== All Hyperparameter Combinations and Validation RMSE ===

RMSE: 7.0367  |  rank: 10, regParam: 0.01
RMSE: 1.5853  |  rank: 10, regParam: 0.1
RMSE: 1.3054  |  rank: 10, regParam: 0.5
RMSE: 4.4477  |  rank: 25, regParam: 0.01
RMSE: 1.4870  |  rank: 25, regParam: 0.1
RMSE: 1.3047  |  rank: 25, regParam: 0.5
RMSE: 3.0035  |  rank: 50, regParam: 0.01
RMSE: 1.4480  |  rank: 50, regParam: 0.1
RMSE: 1.3042  |  rank: 50, regParam: 0.5
RMSE: 2.2127  |  rank: 100, regParam: 0.01
RMSE: 1.4241  |  rank: 100, regParam: 0.1
RMSE: 1.3042  |  rank: 100, regParam: 0.5


## 2.7. Push vectors to ClickHouse

In [10]:
user_indexer_model = index_model.stages[0]
item_indexer_model = index_model.stages[1]

user_labels = user_indexer_model.labels
item_labels = item_indexer_model.labels

user_mapping_sdf = spark.createDataFrame(
    [(i, user_labels[i]) for i in range(len(user_labels))],
    ["user_idx", "user_id"]
)

item_mapping_sdf = spark.createDataFrame(
    [(i, item_labels[i]) for i in range(len(item_labels))],
    ["item_idx", "parent_asin"]
)

user_cf_sdf = (
    model.userFactors
    .withColumnRenamed("id", "user_idx")
    .withColumnRenamed("features", "cf_vector")
    .join(user_mapping_sdf, on="user_idx", how="inner")
    .select("user_id", "cf_vector")
)

item_cf_sdf = (
    model.itemFactors
    .withColumnRenamed("id", "item_idx")
    .withColumnRenamed("features", "cf_vector")
    .join(item_mapping_sdf, on="item_idx", how="inner")
    .select("parent_asin", "cf_vector")
)

In [12]:
queries = [
    f"""KILL MUTATION 
        WHERE database = '{SCHEMA}' 
          AND table = '{per_user_embedding_table}' 
          AND is_done = 0;""",

    f"""ALTER TABLE {SCHEMA}.{per_user_embedding_table}
    DROP INDEX IF EXISTS cf_vector_idx;""",

    f"""ALTER TABLE {SCHEMA}.{per_user_embedding_table}
    DROP COLUMN IF EXISTS cf_vector""",

    f"""ALTER TABLE {SCHEMA}.{per_user_embedding_table}
    ADD COLUMN IF NOT EXISTS cf_vector Array(Float32),
    ADD INDEX IF NOT EXISTS cf_vector_idx cf_vector TYPE vector_similarity(
        'hnsw',             -- Algorithm type
        'cosineDistance',   -- Distance metric
        {ALS_RANK},         -- Vector dimension
        'f32',              -- Vector data type precision ('f32' matches Float32)
        16,                 -- M (Number of connections per node, default is often 16 or 64)
        128                 -- ef_construction (Candidate list size during build)
    );""",

    f"DROP TABLE IF EXISTS {SCHEMA}.users_cf_stage;",
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.users_cf_stage (
        user_id String,
        cf_vector Array(Float32)
    )
    ENGINE = MergeTree
    ORDER BY user_id;"""]

client = connect_clickhouse()

for i, query in enumerate(tqdm(queries, desc="Running the code")):
    while True:
        try:
            client.command(query)
            break
        except:
            time.sleep(60)
        
    print(f"Query {i+1}: Successful")

client.close()

def insert_partition_to_clickhouse(rows):
    import clickhouse_connect

    client = clickhouse_connect.get_client(
        host=CLICKHOUSE_HOST,
        port=CLICKHOUSE_PORT,
        username=CLICKHOUSE_USERNAME,
        password=CLICKHOUSE_PASSWORD,
        database=SCHEMA,
        secure=True
    )

    batch = []
    BATCH_SIZE = 5000

    for row in tqdm(rows, desc="Loading user collaborative filtering vectors"):
        batch.append((row.user_id, row.cf_vector))

        if len(batch) >= BATCH_SIZE:
            client.insert(
                table="users_cf_stage",
                data=batch,
                column_names=["user_id", "cf_vector"]
            )
            batch.clear()

    # Insert remaining
    if batch:
        client.insert(
            table="users_cf_stage",
            data=batch,
            column_names=["user_id", "cf_vector"]
        )

    client.close()

user_cf_sdf.foreachPartition(insert_partition_to_clickhouse)
time.sleep(10)

Running the code:  50%|█████     | 3/6 [00:00<00:00, 22.14it/s]

Query 1: Successful
Query 2: Successful
Query 3: Successful


Running the code:  67%|██████▋   | 4/6 [01:00<00:39, 19.63s/it]

Query 4: Successful
Query 5: Successful


Running the code: 100%|██████████| 6/6 [01:01<00:00, 10.19s/it]

Query 6: Successful



Loading user collaborative filtering vectors: 92391it [00:04, 21991.28it/s]/ 21]
Loading user collaborative filtering vectors: 102909it [00:05, 19334.16it/s]
Loading user collaborative filtering vectors: 102563it [00:05, 19354.45it/s]
Loading user collaborative filtering vectors: 92476it [00:04, 19769.47it/s]/ 21]
Loading user collaborative filtering vectors: 93286it [00:05, 16333.08it/s]
Loading user collaborative filtering vectors: 102939it [00:05, 19650.23it/s]
Loading user collaborative filtering vectors: 102717it [00:05, 19482.08it/s] 21]
Loading user collaborative filtering vectors: 92788it [00:05, 15528.63it/s]
Loading user collaborative filtering vectors: 93078it [00:05, 16324.83it/s]
Loading user collaborative filtering vectors: 92513it [00:06, 15313.31it/s]/ 21]
Loading user collaborative filtering vectors: 92882it [00:06, 15267.07it/s]
Loading user collaborative filtering vectors: 92732it [00:05, 15668.16it/s]/ 21]
Loading user collaborative filtering vectors: 102565it [00:

In [13]:
queries = [
    # 1. Kill any stuck background jobs
    f"""KILL MUTATION 
        WHERE database = '{SCHEMA}' 
          AND table = '{per_item_embedding_table}' 
          AND is_done = 0;""",
          
    # 2. Drop the index AND the column together, and FORCE Python to wait until it finishes
    f"""ALTER TABLE {SCHEMA}.{per_item_embedding_table}
        DROP INDEX IF EXISTS cf_vector_idx,
        DROP COLUMN IF EXISTS cf_vector
        SETTINGS mutations_sync = 2;""",

    # 3. Recreate the column
    f"""ALTER TABLE {SCHEMA}.{per_item_embedding_table}
    ADD COLUMN IF NOT EXISTS cf_vector Array(Float32),
    ADD INDEX IF NOT EXISTS cf_vector_idx cf_vector TYPE vector_similarity(
        'hnsw',             -- Algorithm type
        'cosineDistance',   -- Distance metric
        {ALS_RANK},         -- Vector dimension
        'f32',              -- Vector data type precision ('f32' matches Float32)
        16,                 -- M (Number of connections per node, default is often 16 or 64)
        128                 -- ef_construction (Candidate list size during build)
    );""",

    # 4. Reset the staging table
    f"DROP TABLE IF EXISTS {SCHEMA}.items_cf_stage;",
    f"""CREATE TABLE IF NOT EXISTS {SCHEMA}.items_cf_stage (
        parent_asin String,
        cf_vector Array(Float32)
    )
    ENGINE = MergeTree
    ORDER BY parent_asin;"""
]

client = connect_clickhouse()

for i, query in enumerate(tqdm(queries, desc="Running the code")):
    client.command(query)
    print(f"Query {i+1}: Successful")

client.close()

def insert_partition_to_clickhouse(rows):
    import clickhouse_connect

    client = clickhouse_connect.get_client(
        host=CLICKHOUSE_HOST,
        port=CLICKHOUSE_PORT,
        username=CLICKHOUSE_USERNAME,
        password=CLICKHOUSE_PASSWORD,
        database=SCHEMA,
        secure=True
    )

    batch = []
    BATCH_SIZE = 5000

    for row in tqdm(rows, desc="Loading item collaborative filtering vectors"):
        batch.append((row.parent_asin, row.cf_vector))

        if len(batch) >= BATCH_SIZE:
            client.insert(
                table="items_cf_stage",
                data=batch,
                column_names=["parent_asin", "cf_vector"]
            )
            batch.clear()

    # Insert remaining
    if batch:
        client.insert(
            table="items_cf_stage",
            data=batch,
            column_names=["parent_asin", "cf_vector"]
        )

    client.close()

item_cf_sdf.foreachPartition(insert_partition_to_clickhouse)
time.sleep(10)

Running the code:  60%|██████    | 3/5 [00:00<00:00, 21.03it/s]

Query 1: Successful
Query 2: Successful
Query 3: Successful
Query 4: Successful


Running the code: 100%|██████████| 5/5 [00:00<00:00,  7.28it/s]

Query 5: Successful



Loading item collaborative filtering vectors: 29158it [00:01, 22544.76it/s]/ 22]
Loading item collaborative filtering vectors: 32017it [00:01, 20352.32it/s]/ 22]
Loading item collaborative filtering vectors: 29323it [00:01, 19207.08it/s]
Loading item collaborative filtering vectors: 29216it [00:01, 22811.69it/s]
Loading item collaborative filtering vectors: 29076it [00:01, 17942.96it/s]/ 22]
Loading item collaborative filtering vectors: 29404it [00:01, 17327.30it/s]
Loading item collaborative filtering vectors: 29419it [00:01, 19456.25it/s]/ 22]
Loading item collaborative filtering vectors: 32325it [00:01, 17687.28it/s]
Loading item collaborative filtering vectors: 29274it [00:01, 16622.09it/s]
Loading item collaborative filtering vectors: 29303it [00:01, 21006.19it/s]/ 22]
Loading item collaborative filtering vectors: 6464it [00:00, 17647.77it/s] / 22]
Loading item collaborative filtering vectors: 29201it [00:03, 9102.40it/s]
Loading item collaborative filtering vectors: 29212it [00:

In [15]:
client = connect_clickhouse()

update_queries = [
    # --- A. UPDATE ITEMS ---
    f"CREATE TABLE IF NOT EXISTS {SCHEMA}.items_cf_join (parent_asin String, cf_vector Array(Float32)) ENGINE = Join(ANY, LEFT, parent_asin);",
    f"INSERT INTO {SCHEMA}.items_cf_join SELECT parent_asin, cf_vector FROM {SCHEMA}.items_cf_stage;",
    
    # Notice the new WHERE clause and the SETTINGS mutations_sync = 2
    f"""ALTER TABLE {SCHEMA}.{per_item_embedding_table}
        UPDATE cf_vector = joinGet('{SCHEMA}.items_cf_join', 'cf_vector', parent_asin)
        WHERE joinGet('{SCHEMA}.items_cf_join', 'cf_vector', parent_asin) IS NOT NULL
        SETTINGS mutations_sync = 2;""",
        
    f"DROP TABLE IF EXISTS {SCHEMA}.items_cf_join;",

    # --- B. UPDATE USERS ---
    f"CREATE TABLE IF NOT EXISTS {SCHEMA}.users_cf_join (user_id String, cf_vector Array(Float32)) ENGINE = Join(ANY, LEFT, user_id);",
    f"INSERT INTO {SCHEMA}.users_cf_join SELECT user_id, cf_vector FROM {SCHEMA}.users_cf_stage;",
    
    f"""ALTER TABLE {SCHEMA}.{per_user_embedding_table}
        UPDATE cf_vector = joinGet('{SCHEMA}.users_cf_join', 'cf_vector', user_id)
        WHERE joinGet('{SCHEMA}.users_cf_join', 'cf_vector', user_id) IS NOT NULL
        SETTINGS mutations_sync = 2;""",
        
    f"DROP TABLE IF EXISTS {SCHEMA}.users_cf_join;"
    
    # --- C. (OPTIONAL) CLEAN UP STAGING TABLES ---
    # f"DROP TABLE IF EXISTS {SCHEMA}.items_cf_stage;",
    # f"DROP TABLE IF EXISTS {SCHEMA}.users_cf_stage;"
]

for i, query in enumerate(update_queries):
    client.command(query)
    print(f"Query {i+1}: Successful")
    
client.close()

Query 1: Successful
Query 2: Successful
Query 3: Successful
Query 4: Successful
Query 5: Successful
Query 6: Successful
Query 7: Successful
Query 8: Successful


## 2.8. Generate recommendations with CF

In [ ]:
rec_user_id = "AE22236AFRRSMQIKGG7TPTB75QEA"

user_cf_query = f"""
SELECT cf_vector
FROM {SCHEMA}.{per_user_embedding_table}
WHERE user_id = '{rec_user_id}'
"""
user_cf_df = clickhouse_extract_data_as_df(user_cf_query)
user_cf_vector = user_cf_df.loc[0, "cf_vector"]

if user_cf_vector is None or len(user_cf_vector) == 0:
    raise ValueError("Cold-start user: CF-only recommendation not possible")

# Note: Can't use history column because it stores only the history of the products the user buys in the same category, not all categories. Therefore, Liam created a new table: user_item_interactions
cf_recommendation_query = f"""
SELECT
    raw_products.parent_asin,
    raw_products.title,
    raw_products.product_category,
    rec_products.cosine_similarity
FROM {SCHEMA}.{TABLE_RAW_PRODUCTS} AS raw_products
JOIN (
        SELECT
            v.parent_asin,
            1 - cosineDistance(v.cf_vector, {user_cf_vector}) AS cosine_similarity
        FROM {SCHEMA}.{per_item_embedding_table} v
        WHERE
            notEmpty(v.cf_vector)
            AND {exclude_history_sql_where_clause(rec_user_id)}
        ORDER BY cosine_similarity DESC
        LIMIT 10
    ) AS rec_products ON raw_products.parent_asin = rec_products.parent_asin
ORDER BY rec_products.cosine_similarity DESC
"""
cf_recs = clickhouse_extract_data_as_df(cf_recommendation_query)
cf_recs

,parent_asin,title,product_category,cosine_similarity
0,B009XHW9PK,Skywalker Dual Cable Flexible Screw Clips (Pac...,Electronics,0.952839
1,B004TUT21K,rooCASE Executive Portfolio (Red) Leather Case...,Electronics,0.951909
2,B01M34U6BE,Neoprene Sleeve Case for Boogie Board Jot 8.5 ...,Electronics,0.949973
3,B076CJRVXH,Travel Case for Bose Soundlink Micro Bluetooth...,Electronics,0.949772
4,B0B1JQJ1WC,Angel USA 400 Pound Physician Digital Scale Bo...,Industrial_and_Scientific,0.949648
5,B07ZB7VHCN,228PKS Pegboard Hooks Assortment with Metal Ho...,Industrial_and_Scientific,0.949602
6,B01K5AJ5UU,"Pyle Upgraded Pyle 10"" Portable DVD Player, CD...",Electronics,0.949258
7,B0777V5BRJ,Samsung Galaxy Note 5 SM-N920T 32GB T-Mobile G...,Cell_Phones_and_Accessories,0.948829
8,B07CNRK6GW,LONMEN Water Resistant 12.5-13 Inch Laptop Sle...,Electronics,0.948525
9,B011U2ABK0,ESR Corner/Bumper Protection Smart Cover Case ...,Electronics,0.948291


# 3. Performing Recommendation

We implement different approaches to recommend products:
1. Pure random K products
2. Top K products with highest difference between 4-and-above-star reviews
3. Random K products with 1000 highest difference between 4-and-above-star reviews
4. Content-based
5. Collaborative Filtering
6. Hybrid (CF + Content)
7. LLM-based

In [23]:
# 1. Retrieve user info
def user_info(rec_user_id):
    """
    Extract user embeddings: cf_vector and content vector; also whether they are cold or warm
    """
    rec_user_hybrid_query = f"""
    SELECT
        cf_vector,
        {content_vector_name}
    FROM {SCHEMA}.{per_user_embedding_table}
    WHERE user_id = '{rec_user_id}'
    """
    rec_user_hybrid = clickhouse_extract_data_as_df(rec_user_hybrid_query)

    user_cf_vector = rec_user_hybrid.loc[0, "cf_vector"]
    user_content_vector = rec_user_hybrid.loc[0, content_vector_name]

    is_cold_user = (user_cf_vector is None or len(user_cf_vector) == 0)

    return is_cold_user, user_cf_vector, user_content_vector

# 2. Retrieve candidates before perfoming recommendations for content/CF/hybrid methods
def content_based_candidates(rec_user_id, user_content_vector, k=10):
    candidate_query = f"""
    SELECT
        raw_products.parent_asin,
        raw_products.title,
        raw_products.product_category,
        CAST(NULL AS Nullable(Float32)) AS cf_score,
        CAST(NULL AS Nullable(Float32)) AS predicted_rating,
        rec_products.content_score
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS} AS raw_products
    JOIN (
            SELECT
                v.parent_asin,
                1 - cosineDistance(mixed_vector, {user_content_vector}) AS content_score
            FROM {SCHEMA}.{per_item_embedding_table} v
            WHERE {exclude_history_sql_where_clause(rec_user_id)}
            ORDER BY content_score DESC
            LIMIT {k}
        ) AS rec_products ON raw_products.parent_asin = rec_products.parent_asin
    """
    return clickhouse_extract_data_as_df(candidate_query)

def cf_candidates(rec_user_id, user_cf_vector, user_content_vector, k=10):
    cf_candidates_query = f"""
    SELECT
        raw_products.parent_asin,
        raw_products.title,
        raw_products.product_category,
        rec_products.cf_score,
        rec_products.predicted_rating,
        rec_products.content_score
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS} AS raw_products
    JOIN (
            SELECT
                v.parent_asin,
                1 - cosineDistance(v.cf_vector, {user_cf_vector}) AS cf_score,
                dotProduct(v.cf_vector, {user_cf_vector}) AS predicted_rating,
                1 - cosineDistance(mixed_vector, {user_content_vector}) AS content_score
            FROM {SCHEMA}.{per_item_embedding_table} v
            WHERE
                notEmpty(v.cf_vector)
            AND {exclude_history_sql_where_clause(rec_user_id)}
            ORDER BY cf_score DESC
            LIMIT {k}
        ) AS rec_products ON raw_products.parent_asin = rec_products.parent_asin
    """
    return clickhouse_extract_data_as_df(cf_candidates_query)

def hybrid_candidates(rec_user_id, user_cf_vector, user_content_vector):
    cf_candidates_ = cf_candidates(rec_user_id, user_cf_vector, user_content_vector, k=300)
    content_candidates = content_based_candidates(rec_user_id, user_content_vector, k=200)

    candidates = (
        pd.concat([cf_candidates_, content_candidates])
        .sort_values(by="cf_score", na_position="last", ascending=False)
        .drop_duplicates(subset=["parent_asin"], keep="first")
        .reset_index(drop=True)
    )

    return candidates

def recsys(rec_user_id, is_cold_user, user_cf_vector, user_content_vector, approach, k=10, fallback=None, print_recs=False, verbose=False):

    if is_cold_user:
        if fallback is not None:
            approach = fallback
        elif (fallback is None) & (approach not in ['pure_random', 'top_k', 'random_in_top']):
            fallback = 'top_k'
            approach = fallback

        if print_recs:
            print(f"User is cold. Fall back to {approach} approach.\n")

    if approach == 'pure_random':
        recs_query = f"""
        SELECT parent_asin, title, product_category
        FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
        ORDER BY rand()
        LIMIT {k}
        """
        recs = clickhouse_extract_data_as_df(recs_query)

    elif approach == 'top_k':
        recs_query = f"""
        SELECT
            top_items.parent_asin,
            raw_products.title,
            raw_products.product_category
        FROM
            (SELECT parent_asin
            FROM {SCHEMA}.{TABLE_RECENT_ITEM_REVIEW_SUMMARY}
            ORDER BY high_low_star_diff DESC
            LIMIT {k}) AS top_items
        JOIN {SCHEMA}.{TABLE_RAW_PRODUCTS} AS raw_products on top_items.parent_asin = raw_products.parent_asin
        """
        recs = clickhouse_extract_data_as_df(recs_query)

    elif approach == 'random_in_top':
        recs_query = f"""
        SELECT
            top_items.parent_asin,
            raw_products.title,
            raw_products.product_category
        FROM
            (SELECT parent_asin
            FROM {SCHEMA}.{TABLE_RECENT_ITEM_REVIEW_SUMMARY}
            ORDER BY high_low_star_diff DESC
            LIMIT 1000) AS top_items
        JOIN {SCHEMA}.{TABLE_RAW_PRODUCTS} AS raw_products on top_items.parent_asin = raw_products.parent_asin
        ORDER BY rand()
        LIMIT {k}
        """
        recs = clickhouse_extract_data_as_df(recs_query)

    elif approach == 'content':
        if verbose:
            print("Showing content-based recommendations")
        candidates = content_based_candidates(rec_user_id, user_content_vector, k=k)
        recs = candidates.drop(columns=['cf_score', 'predicted_rating'])

    elif approach == 'cf':
        if verbose:
            print("Showing collaborative filtering recommendations")
        candidates = cf_candidates(rec_user_id, user_cf_vector, user_content_vector, k=k)
        recs = candidates.drop(columns=['content_score'])

    elif approach == 'hybrid':
        if verbose:
            print("Showing hybrid recommendations")
        candidates = hybrid_candidates(rec_user_id, user_cf_vector, user_content_vector)

        # 1. Safely calculate vector confidence (handle cold users)
        cf_conf = np.linalg.norm(user_cf_vector) if not is_cold_user else 0.0
        content_conf = np.linalg.norm(user_content_vector) if user_content_vector is not None else 0.0

        total_conf = cf_conf + content_conf
        ALPHA = cf_conf / total_conf if total_conf > 0 else 0.0

        # 2. Vectorized Hybrid Re-ranking (No more .iterrows!)
        # Fill NaN cf_scores (from content candidates) with 0.0
        candidates["cf_score_filled"] = candidates["cf_score"].fillna(0.0)
        
        # If the user is cold, force ALPHA to 0 so it relies 100% on content_score
        if is_cold_user:
            ALPHA = 0.0

        # Calculate hybrid score for the whole dataframe at once
        candidates["hybrid_score"] = (ALPHA * candidates["cf_score_filled"]) + ((1 - ALPHA) * candidates["content_score"])

        # Clean up the temporary column used for math
        candidates = candidates.drop(columns=["cf_score_filled"])

        recs = candidates.sort_values("hybrid_score", ascending=False).drop(columns=['cf_score', 'content_score']).head(k)

    elif approach == 'LLM':
        pass

    # if print_recs:
    #     print(f"Recommendation of {k} items:")
    #     display(recs)

    #     if not is_cold_user:
    #         print("\nUser Review History")
    #         history_query = f"""
    #             SELECT
    #                 raw_products.product_category AS product_category,
    #                 reviews.rating AS rating,
    #                 COUNT(reviews.rating) AS review_count
    #             FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT['train']} AS reviews
    #             JOIN {SCHEMA}.{TABLE_RAW_PRODUCTS} AS raw_products ON raw_products.parent_asin = reviews.parent_asin
    #             WHERE reviews.user_id = '{rec_user_id}'
    #             GROUP BY 1, 2
    #             ORDER BY review_count DESC
    #         """
    #         display(clickhouse_extract_data_as_df(history_query))
    
    return recs

In [ ]:
rec_user_id = "AGUTZC4GHLTGYHA3KBEDRF6MHB6A" # warm user
is_cold_user, user_cf_vector, user_content_vector = user_info(rec_user_id)

if is_cold_user:
    print(f"User {rec_user_id} is a cold user.\n")
else:
    print(f"User {rec_user_id} is a warm user.\n")

history_query = f"""
                SELECT
                    raw_products.product_category AS product_category,
                    reviews.rating AS rating,
                    COUNT(reviews.rating) AS review_count
                FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT['train']} AS reviews
                JOIN {SCHEMA}.{TABLE_RAW_PRODUCTS} AS raw_products ON raw_products.parent_asin = reviews.parent_asin
                WHERE reviews.user_id = '{rec_user_id}'
                GROUP BY 1, 2
                ORDER BY review_count DESC
            """
display(clickhouse_extract_data_as_df(history_query))

User AGUTZC4GHLTGYHA3KBEDRF6MHB6A is a warm user.



,product_category,rating,review_count
0,Electronics,5,153
1,Toys_and_Games,5,141
2,Arts_Crafts_and_Sewing,5,41
3,Cell_Phones_and_Accessories,5,36
4,Baby_Products,5,29
5,Musical_Instruments,5,11
6,Industrial_and_Scientific,5,7
7,Baby_Products,1,1


In [31]:
recsys(rec_user_id, is_cold_user, user_cf_vector, user_content_vector, approach='content', print_recs=True, verbose=True)

Showing content-based recommendations


,parent_asin,title,product_category,content_score
0,B07Q7GJSN9,"Punswan for Airpods 1 & 2 Charging Case,3D Cut...",Electronics,0.978898
1,B071VY7NF4,"Invisible Ink Pen, Spy Pen Secret Message Writ...",Toys_and_Games,0.980034
2,B07DXPTCXP,"Fidget Toys Set, 36 Pack Sensory Tools Bundle ...",Toys_and_Games,0.980389
3,B084TB5S5C,PHOUPHO Hobby-rc-Cars,Toys_and_Games,0.979227
4,B08MF3GG3K,Clear Accessories Kits for Adults & Kids,Toys_and_Games,0.979845
5,B093GNV9M4,"Fidget Toys, Push Bubble Fidgets Sensory Toy, ...",Toys_and_Games,0.980426
6,B093K76HXY,"cinsey Fidget Toy Set, Cheap Sensory Toys Pack...",Toys_and_Games,0.979816
7,B08CZ4HJC1,Dysania HTV Heat Transfer Vinyl Bundle - 40 Pa...,Arts_Crafts_and_Sewing,0.979225
8,B0BWR96D29,"Flying Orb Ball, Hover Ball Hand Controlled Bo...",Toys_and_Games,0.979017
9,B0BZPNZST2,VTech KidiZoom Duo DX Digital Selfie Camera wi...,Toys_and_Games,0.978813


In [26]:
recsys(rec_user_id, is_cold_user, user_cf_vector, user_content_vector, approach='cf', print_recs=True, verbose=True)

Showing collaborative filtering recommendations


,parent_asin,title,product_category,cf_score,predicted_rating
0,B08L7LFCBV,"Permanent Smart Vinyl, 5.5”x 120” Bulk Roll fo...",Arts_Crafts_and_Sewing,0.997510,4.526268
1,B0BNPRYRKJ,"DIY Candle Making Kit, Beeswax Candles Making ...",Arts_Crafts_and_Sewing,0.997510,4.526268
2,B08L3PH4X2,Mini 65W USB C Charger for MacBook Pro Air Fas...,Cell_Phones_and_Accessories,0.997510,4.526268
3,B09H74YQ7W,"2” Caster Wheels,Set of 4,Heavy Duty Swivel Ca...",Industrial_and_Scientific,0.997510,4.526268
4,B08RYSB432,"PERLECARE Inflatable Pool, Swimming Pool for K...",Toys_and_Games,0.995775,4.614187
5,B09MRWFB84,Charging Station for Multiple Devices - 6 Port...,Cell_Phones_and_Accessories,0.997510,4.526268
6,B09YH85CC1,"MAGICYOYO Responsive Yoyo T9 Shark-Blue, Metal...",Toys_and_Games,0.994454,4.423638
7,B0B59343Z4,"Clear Balloons , Birthday Crystal Bubble Bobo ...",Toys_and_Games,0.997510,4.526268
8,B08FX53ZQG,OBSEDE 81Pcs Jewelry Making Kit with 36 PCS Wo...,Arts_Crafts_and_Sewing,0.995179,4.207165
9,B0C9T99NQD,Wood Whittling Kit with Basswood Wood Blocks G...,Arts_Crafts_and_Sewing,0.997510,4.526268


In [27]:
recsys(rec_user_id, is_cold_user, user_cf_vector, user_content_vector, approach='hybrid', print_recs=True, verbose=True)

Showing hybrid recommendations


,parent_asin,title,product_category,predicted_rating,hybrid_score
4,B0C9T99NQD,Wood Whittling Kit with Basswood Wood Blocks G...,Arts_Crafts_and_Sewing,4.526268,0.988076
11,B07WJQT81G,LEGO Star Wars Snowspeeder 75268 Starship Toy ...,Toys_and_Games,4.776382,0.987591
18,B07DK4B1N7,"Race Tracks for Boys, Car Toys STEM Track Adve...",Toys_and_Games,4.420209,0.987316
6,B0B59343Z4,"Clear Balloons , Birthday Crystal Bubble Bobo ...",Toys_and_Games,4.526268,0.987257
181,B01AAQOYDM,ATB 3 Decks Spanish Playing Cards Baraja Espan...,Toys_and_Games,4.642575,0.986710
158,B07NDXKXTD,Hot Wheels iD Tesla Model S,Toys_and_Games,4.583111,0.986106
52,B08BG6CVQD,CuberSpeed MFJS Moyu RS3 M 2020 Flagship 3x3 S...,Toys_and_Games,5.023186,0.986038
133,B08P7JNQ62,MTMETY 10x7Ft Galaxy Backdrop Universe Backdro...,Electronics,4.762099,0.986032
123,B07Z6Y547C,Banpresto Sword Art Online Integral Factor Esp...,Toys_and_Games,4.609902,0.985800
66,B07JFY2QHH,Amazon Basics Urban Laptop and Tablet Case Bag...,Electronics,4.564945,0.985473


## Fallback

In [ ]:
rec_user_id = "AE22222X4BO3JQVOMEHHRPVRTB6A" # cold user
is_cold_user, user_cf_vector, user_content_vector = user_info(rec_user_id)

if is_cold_user:
    print(f"User {rec_user_id} is a cold user.\n")
else:
    print(f"User {rec_user_id} is a warm user.\n")

recsys(rec_user_id, is_cold_user, user_cf_vector, user_content_vector, approach='hybrid', print_recs=True)

User AE22222X4BO3JQVOMEHHRPVRTB6A is a cold user.

User is cold. Fall back to top_k approach.



,parent_asin,title,product_category
0,B0791TX5P5,Fire TV Stick streaming device with Alexa buil...,Electronics
1,B07GZFM1ZM,Fire TV Stick 4K streaming device with Alexa V...,Electronics
2,B07HZLHPKP,"Echo Show 5 (1st Gen, 2019 release) -- Smart d...",Electronics
3,B07KTYJ769,"Amazon Smart Plug, for home automation, Works ...",Electronics
4,B07456BG8N,Echo (2nd Generation) - Smart speaker with Ale...,Electronics
5,B075X8471B,"Fire TV Stick with Alexa Voice Remote, streami...",Electronics
6,B07H65KP63,"Echo Dot (3rd Gen, 2018 release) - Smart speak...",Electronics
7,B07PHQ93TV,"Fire HD 10 Tablet (10.1"" 1080p full HD display...",Electronics
8,B08XPWDSWW,TOZO T10 Bluetooth 5.3 Wireless Earbuds with W...,Electronics
9,B01K8B8YA8,Echo Dot (2nd Generation) - Smart speaker with...,Electronics


In [29]:
recsys(rec_user_id, is_cold_user, user_cf_vector, user_content_vector, approach='hybrid', fallback='pure_random', print_recs=True)

User is cold. Fall back to pure_random approach.



,parent_asin,title,product_category
0,B0775TNGMY,"iPhone 8 Case, Silverback Girls Bling Glitter ...",Cell_Phones_and_Accessories
1,B00JR9UYME,Magic The Gathering - Deicide (/165) - Journey...,Toys_and_Games
2,B08F7FJRRQ,KIMROO 24 Pcs 45cm/17inch Hair Curlers Styling...,All_Beauty
3,B082XYTGDS,Hotwin 200 pcs Square Rectangle Foam Solvent C...,Cell_Phones_and_Accessories
4,B012V6EO2A,CreateDesigned The Little Mermaid Phone Cases ...,Cell_Phones_and_Accessories
5,B01B14X7QW,Learning Resources Magnetic Counting Vehicle P...,Toys_and_Games
6,B09H4M9Z7R,Kids Camera for Boys Girls - Selfie Camera for...,Electronics
7,B017V21O3Q,Texas Fall Ornament,Handmade_Products
8,B07CWGSR5V,DJI Phantom 4 Pro Obsidian Service Part - Land...,Toys_and_Games
9,B08JKX93VP,"Industrial Curtain Rod, Curtain Rods for Windo...",Electronics


# 4. Evaluation

In [48]:
def evaluate_all_metrics(user_id_list, test_df, approach, k_list, rating_threshold=4.0):
    max_k = max(k_list)
    
    # 1. Prepare Ground Truth & get Total Relevant items per user
    relevant = test_df.filter(
        (col("rating") >= rating_threshold) & 
        (col("user_id").isin(user_id_list))
    ).select("user_id", "parent_asin").withColumn("relevant", lit(1))
    
    user_relevant_counts = relevant.groupBy("user_id").count().withColumnRenamed("count", "total_relevant")

    # 2. Fetch recommendations ONCE per user for max_k
    all_user_recs = []
    for rec_user_id in user_id_list:
        is_cold_user, user_cf_vector, user_content_vector = user_info(rec_user_id)
        
        recs = recsys(rec_user_id, is_cold_user, user_cf_vector, user_content_vector, approach=approach, k=max_k)

        if not recs.empty:
            recs['user_id'] = rec_user_id
            recs['rank'] = range(1, len(recs) + 1) 
            all_user_recs.append(recs[['user_id', 'parent_asin', 'rank']])

    # Fallback if no recommendations were generated at all
    if not all_user_recs:
        empty_results = {}
        for k in k_list:
            empty_results.update({f'P@{k}': 0.0, f'HR@{k}': 0.0, f'NDCG@{k}': 0.0, f'Cov@{k}': 0})
        return empty_results

    # 3. Combine and convert to Spark
    combined_pdf = pd.concat(all_user_recs, ignore_index=True)
    recs_spark = spark.createDataFrame(combined_pdf)

    # 4. GLOBAL METRIC: Item Coverage @ K
    cov_exprs = [countDistinct(when(col("rank") <= k, col("parent_asin"))).alias(f"cov_{k}") for k in k_list]
    coverage_row = recs_spark.agg(*cov_exprs).first()

    # 5. Join with ground truth for accuracy metrics
    recs_with_relevance = recs_spark.join(
        relevant, ["user_id", "parent_asin"], "left"
    ).fillna(0, subset=["relevant"])

    # 6. Calculate Precision (sum), Hit Rate (max), and DCG components per user
    user_agg_exprs = []
    for k in k_list:
        # For Precision: Sum of relevant items in top K
        rel_at_k = spark_sum(when(col("rank") <= k, col("relevant")).otherwise(0)).alias(f"rel_at_{k}")
        
        # For Hit Rate: Max is 1 if at least one relevant item is in top K, else 0
        hit_at_k = spark_max(when(col("rank") <= k, col("relevant")).otherwise(0)).alias(f"hit_{k}")
        
        # For DCG@K: Sum of (1 / log2(rank + 1)) for relevant items in top K
        dcg_at_k = spark_sum(
            when((col("rank") <= k) & (col("relevant") == 1), 1.0 / log2(col("rank") + 1.0)).otherwise(0.0)
        ).alias(f"dcg_{k}")
        
        user_agg_exprs.extend([rel_at_k, hit_at_k, dcg_at_k])

    user_base_metrics = recs_with_relevance.groupBy("user_id").agg(*user_agg_exprs)

    # 7. Join with total_relevant to calculate IDCG, NDCG, and final Precision
    user_metrics = user_base_metrics.join(user_relevant_counts, "user_id", "left").fillna(0)

    for k in k_list:
        # Precision@K = (relevant items in top K) / K
        user_metrics = user_metrics.withColumn(f"precision_{k}", col(f"rel_at_{k}") / lit(k))

        # IDCG@K sequence generation
        idcg_expr = expr(f"aggregate(transform(sequence(1, least({k}, cast(total_relevant as int))), i -> 1.0 / log2(i + 1.0)), 0D, (acc, x) -> acc + x)")
        
        # NDCG calculation
        user_metrics = user_metrics.withColumn(
            f"ndcg_{k}", 
            when(idcg_expr > 0, col(f"dcg_{k}") / idcg_expr).otherwise(0.0)
        )

    # 8. Final Aggregation: Average across all valid test users
    final_avg_exprs = []
    for k in k_list:
        final_avg_exprs.append(spark_avg(f"precision_{k}").alias(f"P@{k}"))
        final_avg_exprs.append(spark_avg(f"hit_{k}").alias(f"HR@{k}"))
        final_avg_exprs.append(spark_avg(f"ndcg_{k}").alias(f"NDCG@{k}"))

    avg_row = user_metrics.agg(*final_avg_exprs).first()

    # 9. Format output dictionary
    results = {}
    for k in k_list:
        results[f'P@{k}'] = avg_row[f"P@{k}"] if avg_row else 0.0
        results[f'HR@{k}'] = avg_row[f"HR@{k}"] if avg_row else 0.0
        results[f'NDCG@{k}'] = avg_row[f"NDCG@{k}"] if avg_row else 0.0
        results[f'Cov@{k}'] = coverage_row[f"cov_{k}"] if coverage_row else 0
        
    return results

def output_evaluation(k_list, all_approaches, user_type, test_user_count):
    all_results = []

    print('Evaluating Precision, Hit Rate, NDCG, and Item Coverage at K...')

    if user_type == 'warm':
        user_id_list_query = f"""
        SELECT user_id
        FROM {SCHEMA}.{per_user_embedding_table}
        WHERE notEmpty(cf_vector) AND notEmpty({content_vector_name})
        AND user_id in (SELECT user_id FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT['valid']})
        AND user_id in (SELECT user_id FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT['train']})
        LIMIT {test_user_count}
        """
    elif user_type == 'cold':
        user_id_list_query = f"""
        SELECT user_id
        FROM {SCHEMA}.{per_user_embedding_table}
        WHERE empty(cf_vector) AND empty({content_vector_name})
        AND user_id in (SELECT user_id FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT['valid']})
        LIMIT {test_user_count}
        """

    user_id_list = list(clickhouse_extract_data_as_df(user_id_list_query)['user_id'].values)

    for approach in all_approaches:
        print(f"Running evaluation for approach: {approach}")
        
        results_dict = evaluate_all_metrics(
            user_id_list=user_id_list, 
            test_df=valid_5core, 
            approach=approach, 
            k_list=k_list
        )
        
        results_dict['Approach'] = approach
        all_results.append(results_dict)

    # Create final DataFrame
    final_df = pd.DataFrame(all_results)

    # Order the columns nicely
    col_order = ['Approach']
    for k in k_list:
        col_order.extend([f'P@{k}', f'HR@{k}', f'NDCG@{k}', f'Cov@{k}'])
        
    final_df = final_df[col_order]
    final_df.to_csv(f'{user_type}_{test_user_count}_user_evaluation.csv', index=False)

    return final_df

In [ ]:
k_list = [10, 20, 50]
all_approaches = ['pure_random', 'top_k', 'random_in_top', 'content', 'cf', 'hybrid']
test_user_count = 100

output_evaluation(k_list=k_list, all_approaches=all_approaches, user_type='warm', test_user_count=test_user_count)

Evaluating Precision, Hit Rate, NDCG, and Item Coverage at K...
Running evaluation for approach: pure_random


Running evaluation for approach: top_k


Running evaluation for approach: random_in_top


Running evaluation for approach: content


Running evaluation for approach: cf


Running evaluation for approach: hybrid


,Approach,P@10,HR@10,NDCG@10,Cov@10,P@20,HR@20,NDCG@20,Cov@20,P@50,HR@50,NDCG@50,Cov@50
0,pure_random,0.0,0.0,0.0,1000,0.0000,0.00,0.000000,2000,0.0000,0.00,0.000000,5000
1,top_k,0.0,0.0,0.0,16,0.0000,0.00,0.000000,27,0.0004,0.02,0.001716,50
2,random_in_top,0.0,0.0,0.0,638,0.0000,0.00,0.000000,870,0.0000,0.00,0.000000,992
3,content,0.0,0.0,0.0,821,0.0000,0.00,0.000000,1644,0.0000,0.00,0.000000,4037
4,cf,0.0,0.0,0.0,959,0.0000,0.00,0.000000,1882,0.0002,0.01,0.000685,4683
5,hybrid,0.0,0.0,0.0,923,0.0005,0.01,0.002789,1781,0.0002,0.01,0.002789,4291


In [49]:
k_list = [10, 20, 50]
all_approaches = ['pure_random', 'top_k', 'random_in_top', 'content', 'cf', 'hybrid']
test_user_count = 100

output_evaluation(k_list=k_list, all_approaches=all_approaches[:3], user_type='cold', test_user_count=test_user_count)

# When multiple products share the same high_low_star_diff score (ties), ClickHouse resolves them non-deterministically. So each of the 100 users gets slightly different items at the boundary of the top-K, resulting in 16 unique items instead of 10.

Evaluating Precision, Hit Rate, NDCG, and Item Coverage at K...
Running evaluation for approach: pure_random


Running evaluation for approach: top_k


Running evaluation for approach: random_in_top


,Approach,P@10,HR@10,NDCG@10,Cov@10,P@20,HR@20,NDCG@20,Cov@20,P@50,HR@50,NDCG@50,Cov@50
0,pure_random,0.0,0.0,0.0,1000,0.0,0.0,0.0,2000,0.0,0.0,0.0,4997
1,top_k,0.0,0.0,0.0,16,0.0,0.0,0.0,27,0.0,0.0,0.0,50
2,random_in_top,0.0,0.0,0.0,618,0.0,0.0,0.0,870,0.0,0.0,0.0,996


## Popularity Bias

In [35]:
def popularity_bias(test_df, test_user_count):
    user_id_list_query = f"""
    SELECT user_id
    FROM {SCHEMA}.{per_user_embedding_table}
    WHERE notEmpty(cf_vector) AND notEmpty({content_vector_name})
    AND user_id in (SELECT user_id FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT['valid']})
    AND user_id in (SELECT user_id FROM {SCHEMA}.{TABLE_5CORE_TIMESTAMP_W_HIS_DICT['train']})
    LIMIT {test_user_count}
    """

    user_id_list = list(clickhouse_extract_data_as_df(user_id_list_query)['user_id'].values)

    # 1. Prepare Ground Truth & get Total Relevant items per user
    relevant = test_df.filter(
        (col("rating") >= 4.0) & 
        (col("user_id").isin(user_id_list))
    ).select("user_id", "parent_asin")

    # 2. Fetch recommendations ONCE per user for max_k
    all_user_recs = []
    for rec_user_id in user_id_list:
        is_cold_user, user_cf_vector, user_content_vector = user_info(rec_user_id)
        
        recs = recsys(rec_user_id, is_cold_user, user_cf_vector, user_content_vector, approach='cf', k=10)

        if not recs.empty:
            recs['user_id'] = rec_user_id
            all_user_recs.append(recs[['user_id', 'parent_asin']])

    # 3. Combine and convert to Spark
    combined_pdf = pd.concat(all_user_recs, ignore_index=True)

    # 4. Select top 100 products
    top_100_query = f"""
        SELECT parent_asin
        FROM {SCHEMA}.{TABLE_RECENT_ITEM_REVIEW_SUMMARY}
        ORDER BY high_low_star_diff DESC
        LIMIT 100
        """
    top_100 = clickhouse_extract_data_as_df(top_100_query)

    # 5. Calculate Popularity Bias
    # Convert top 100 items to a set for O(1) lookups
    popular_items_set = set(top_100['parent_asin'].values)
    
    # Flag which recommended items are in the popular set
    combined_pdf['is_popular'] = combined_pdf['parent_asin'].apply(lambda x: x in popular_items_set)
    
    # Calculate the ratio: (Count of popular items recommended) / (Total items recommended)
    total_recommendations = len(combined_pdf)
    popular_recommendations_count = combined_pdf['is_popular'].sum()
    
    popularity_bias_score = popular_recommendations_count / total_recommendations
    
    print("Total recommendations for 100 users:", total_recommendations)
    print("Number of top-100 items in the recommendations:", popular_recommendations_count)
    print("Popularity Bias Score:", popularity_bias_score)

popularity_bias(valid_5core, 100)

Total recommendations for 100 users: 1000
Number of top-100 items in the recommendations: 5
Popularity Bias Score: 0.005


# Clean up

In [18]:
# Clean up: stop Spark session
spark.stop()
print("Spark session stopped!")

Spark session stopped!
